# Paper #33 Implementation: Solar Modulation of Cosmic Rays / 우주선의 태양 조절

Based on Potgieter, M. S. (2013), *Living Reviews in Solar Physics*, DOI: 10.12942/lrsp-2013-3.

이 노트북은 우주선 태양 조절의 핵심 모델을 구현한다:
This notebook implements core solar modulation models:

1. Force-field 근사 / Force-field approximation
2. 변조 포텐셜 Φ에 따른 양성자 스펙트럼 / Proton spectrum at different Φ
3. 태양 주기 동안의 Φ(t) 시계열 / Φ(t) time series across solar cycle
4. A>0 vs A<0 극성 이력현상 (hysteresis) / Polarity hysteresis
5. 1D Parker 수송 방정식 유한차분 / 1D Parker transport finite difference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

M_PROTON_MEV = 938.272

## 1. Force-Field Approximation / Force-field 근사

Gleeson & Axford (1967/68) force-field 해:

$$
J(T) = J_{\rm LIS}(T + \Phi) \cdot \frac{T(T + 2 m_p c^2)}{(T + \Phi)(T + \Phi + 2 m_p c^2)}
$$

- $T$: 지구에서의 운동에너지 (MeV)
- $T + \Phi$: heliopause에서의 운동에너지 (LIS 스펙트럼 참조점)
- $\Phi$: 변조 포텐셜 (MV, 태양 활동에 따라 ~300 MV (극소기) ~ 1500 MV (극대기))

LIS 모델로 Burger et al. (2000) 근사: $J_{\rm LIS}(T) = 1.9 \times 10^4 \cdot T^{-2.78}/(1 + 0.4866 \cdot T^{-2.51})$ (양성자, T in GeV).

In [ ]:
def proton_LIS(T_MeV):
    """Burger et al. 2000 approx. of local interstellar proton spectrum.

    Args:
        T_MeV: Kinetic energy in MeV (can be array).

    Returns:
        J_LIS in particles / (m^2 s sr MeV).
    """
    T_GeV = T_MeV / 1000.0
    return 1.9e4 * T_GeV ** (-2.78) / (1.0 + 0.4866 * T_GeV ** (-2.51)) / 1000.0

def force_field(T_MeV, Phi_MV):
    """Modulated proton differential intensity at Earth.

    Args:
        T_MeV: Kinetic energy at Earth (MeV).
        Phi_MV: Modulation potential in MV.

    Returns:
        J(T) at Earth in same units as proton_LIS.
    """
    mp = M_PROTON_MEV
    T_LIS = T_MeV + Phi_MV
    J_LIS = proton_LIS(T_LIS)
    factor = (T_MeV * (T_MeV + 2 * mp)) / (T_LIS * (T_LIS + 2 * mp))
    return J_LIS * factor

T_grid = np.logspace(1, 5, 200)
plt.figure()
plt.loglog(T_grid, proton_LIS(T_grid), 'k--', lw=2, label='LIS (Φ = 0)')
for Phi, color in [(300, 'blue'), (500, 'green'), (700, 'orange'),
                   (1000, 'red'), (1500, 'purple')]:
    plt.loglog(T_grid, force_field(T_grid, Phi), color=color, label=f'Φ = {Phi} MV')
plt.xlabel('Kinetic energy T (MeV)')
plt.ylabel('J(T) (particles/m²/s/sr/MeV)')
plt.title('Force-field modulated proton spectrum / Force-field 조절 양성자 스펙트럼')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

print('Modulation factor at T = 100 MeV:')
for Phi in [300, 500, 700, 1000, 1500]:
    ratio = force_field(100, Phi) / proton_LIS(100)
    print(f'  Φ = {Phi:4d} MV: J(100 MeV)/LIS = {ratio:.3e}')
print('\nModulation factor at T = 1 GeV:')
for Phi in [300, 500, 700, 1000, 1500]:
    ratio = force_field(1000, Phi) / proton_LIS(1000)
    print(f'  Φ = {Phi:4d} MV: J(1 GeV)/LIS = {ratio:.3f}')

## 2. Solar Cycle Φ(t) Time Series / 태양 주기 Φ(t) 시계열

두 번의 11년 주기를 모사. 각 주기는 극대기에서 Φ ~1300 MV, 극소기에서 Φ ~350 MV 사이를 변동.
Simulate two 11-year cycles. Φ peaks at ~1300 MV at maximum, drops to ~350 MV at minimum.

In [ ]:
def Phi_of_t(years, Phi_min=350.0, Phi_max=1300.0, period=11.0, offset=0.0):
    """Synthetic modulation potential as a function of time."""
    phase = 2 * np.pi * (years - offset) / period
    activity = 0.5 * (1 - np.cos(phase))
    return Phi_min + (Phi_max - Phi_min) * activity

years = np.linspace(0, 22, 2200)
Phi_t = Phi_of_t(years)

counts_relative = np.array([force_field(100.0, p) / force_field(100.0, 350.0) for p in Phi_t])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
ax1.plot(years, Phi_t, 'b-', lw=1.5)
ax1.set_ylabel('Modulation potential Φ (MV)')
ax1.set_title('Solar modulation potential over two cycles / 두 주기의 변조 포텐셜')
ax1.grid(alpha=0.3)

ax2.plot(years, counts_relative, 'r-', lw=1.5)
ax2.set_xlabel('Time (years)')
ax2.set_ylabel('Relative CR flux @100 MeV')
ax2.set_title('Anti-correlated CR flux / 반상관된 우주선 플럭스')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Φ range:           [{Phi_t.min():.0f}, {Phi_t.max():.0f}] MV')
print(f'CR flux range:     [{counts_relative.min():.3f}, {counts_relative.max():.3f}] (normalized to min Φ)')

## 3. A>0 vs A<0 Polarity Hysteresis / 극성 이력현상

Drift effects cause different modulation shapes in A>0 vs A<0 cycles:
- A>0: 양성자 극 방향 유입, HCS 유출 → 평평한 극소기
- A<0: 양성자 HCS 유입 → 뾰족한 극소기

Observed NM count vs Φ draws a closed hysteresis loop.

In [ ]:
t_22yr = np.linspace(0, 22, 2200)
Phi_22 = Phi_of_t(t_22yr)
polarity_A = np.where(t_22yr < 11, +1, -1)

count_base = np.array([force_field(100.0, p) / force_field(100.0, 350.0) for p in Phi_22])
drift_asym = 0.05 * polarity_A * np.sin(np.pi * (t_22yr % 11) / 11)
count_obs = count_base * (1 + drift_asym)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(t_22yr, count_obs, 'b-', lw=1.5)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Relative CR flux @100 MeV')
ax1.set_title('CR flux: A>0 (0-11 yr) vs A<0 (11-22 yr)')
ax1.grid(alpha=0.3)
ax1.axvline(11, color='r', linestyle='--', alpha=0.5, label='Polarity reversal')
ax1.legend()

mask_A_pos = polarity_A > 0
mask_A_neg = polarity_A < 0
ax2.plot(Phi_22[mask_A_pos], count_obs[mask_A_pos], 'b-', lw=2, label='A>0')
ax2.plot(Phi_22[mask_A_neg], count_obs[mask_A_neg], 'r-', lw=2, label='A<0')
ax2.set_xlabel('Modulation potential Φ (MV)')
ax2.set_ylabel('Relative CR flux @100 MeV')
ax2.set_title('Hysteresis loop / 이력현상 loop')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 1D Parker Transport Equation / 1D Parker 수송 방정식

단순화된 spherically symmetric Parker 방정식 (고정 에너지):
$$
\frac{\partial f}{\partial t} = -V \frac{\partial f}{\partial r} + \kappa \left(\frac{\partial^2 f}{\partial r^2} + \frac{2}{r}\frac{\partial f}{\partial r}\right)
$$
경계: $f(R_{\rm HP}) = f_{\rm LIS}$, $\partial f / \partial r = 0$ at $r \to 0$.

In [ ]:
def solve_parker_1d(V_kms=400.0, kappa_m2s=1e22, R_HP_AU=120.0, f_LIS=1.0, n_r=200, n_steps=20000):
    """Solve steady-state 1D Parker transport (convection + radial diffusion)."""
    AU = 1.496e11
    r = np.linspace(0.1, R_HP_AU, n_r) * AU
    dr = r[1] - r[0]
    V = V_kms * 1000.0
    dt = min(0.3 * dr ** 2 / kappa_m2s, 0.3 * dr / V)

    f = np.ones(n_r) * f_LIS * np.exp(-V * (R_HP_AU * AU - r) / kappa_m2s)
    f[-1] = f_LIS

    for _ in range(n_steps):
        df_dr = np.zeros_like(f)
        df_dr[1:-1] = (f[2:] - f[:-2]) / (2 * dr)
        d2f = np.zeros_like(f)
        d2f[1:-1] = (f[2:] - 2 * f[1:-1] + f[:-2]) / dr ** 2
        rhs = -V * df_dr + kappa_m2s * (d2f + (2.0 / r) * df_dr)
        f = f + dt * rhs
        f[0] = f[1]
        f[-1] = f_LIS
        f = np.clip(f, 0, f_LIS * 1.1)
    return r / AU, f

plt.figure()
for kappa in [3e21, 1e22, 3e22, 1e23]:
    r_au, f = solve_parker_1d(V_kms=400, kappa_m2s=kappa, R_HP_AU=120, n_steps=30000)
    plt.plot(r_au, f, label=f'κ = {kappa:.0e} m²/s')
plt.xlabel('Radius (AU)')
plt.ylabel('f(r) / f_LIS')
plt.title('Steady-state Parker transport (V=400 km/s) / 정상상태 Parker 수송')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

r_au, f = solve_parker_1d(V_kms=400, kappa_m2s=1e22, R_HP_AU=120, n_steps=30000)
earth_idx = np.argmin(np.abs(r_au - 1.0))
print(f'At 1 AU, κ=1e22 m²/s: f/f_LIS = {f[earth_idx]:.3f}')
print(f'Effective modulation: ln(LIS/f) = {-np.log(max(f[earth_idx], 1e-6)):.2f}')

## Summary / 요약

1. **Force-field** — 저에너지(<100 MeV) 변조 수십~수백배, 고에너지(>10 GeV) 거의 없음.
2. **Φ(t)** — 극대기 ~1300 MV, 극소기 ~350 MV; CR flux 반상관.
3. **Hysteresis** — A>0 vs A<0은 drift로 서로 다른 모양; Φ-count 궤적은 loop.
4. **Parker 1D** — κ 클수록 약한 변조.

### References
- Potgieter, M. S. (2013). *Solar Modulation of Cosmic Rays*. LRSP 10, 3.
- Parker, E. N. (1965). *The passage of energetic charged particles through interplanetary space*. Planet. Space Sci., 13, 9.
- Gleeson, L. J. & Axford, W. I. (1968). *Solar modulation of galactic cosmic rays*. ApJ, 154, 1011.
- Burger, R. A. et al. (2000). *Rigidity dependence of cosmic ray proton latitudinal gradients*. JGR, 105, 27447.